In [31]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
from IPython.display import Markdown, display
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
deepseek = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com/v1")

In [11]:
def get_weather(location: str) -> str:
    """Simulates a live weather database lookup."""
    # Hardcode a rainy response for testing to see if it advises an umbrella!
    data = {"location": location, "temperature_c": 14, "condition": "Heavy Rain"}
    return json.dumps(data)

def search_hotels(location: str, max_price: int) -> str:
    """Simulates a hotel availability search."""
    data = [
        {"name": "Zurich Capsule Stay", "price_per_night": 65},
        {"name": "Swiss Business Inn", "price_per_night": 130},
        {"name": "Hayat Luxe Tokyo", "price_per_night": 450}
    ]
    # Filter out hotels above the budget
    available = [h for h in data if h["price_per_night"] <= max_price]
    return json.dumps(available)

In [9]:
get_weather_json ={
    "name": "get_weather",
    "description": "Get the current weather condition and temperature for a specific location to determine what clothing or gear to pack.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "The city and country to check the weather for, e.g., Tokyo, Japan"}
        },
        "required": ["location"],
        "additionalProperties": False
    }
}

search_hotels_json = {
    "name": "search_hotels",
    "description": "Search for a list of available hotels in a specific destination that fall within or under a maximum price budget.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "The city or destination area to search for hotels, e.g., Tokyo, Japan"},
            "max_price": {"type": "integer", "description": "The maximum price per night allowed in USD (numbers only, e.g., 150)"}
        },
        "required": ["location", "max_price"],
        "additionalProperties": False
    }
}

In [6]:
tools = [{"type": "function", "function": get_weather_json},
         {"type": "function", "function": search_hotels_json}]

In [7]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [14]:
def travel_agent(messages):
    done = False
    while not done:
        response = deepseek.chat.completions.create(model="deepseek-v4-flash", messages=messages, tools=tools, reasoning_effort="low")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [22]:
system_message = """
You are an expert Autonomous Travel Concierge Agent. You operate in two distinct modes depending on the user's input:

1. TOOL MODE (Real-time data): If the user asks for specific, real-time information such as current weather conditions, packing recommendations based on forecasts, or hotel availability/pricing within a budget, you MUST use your tools. Never guess or invent real-time data.
2. CHAT MODE (General knowledge): If the user asks general, historical, cultural, or open-ended travel questions (e.g., local food recommendations, tipping etiquette, language phrases, or general itineraries), do NOT call any tools. Answer immediately using your vast internal knowledge.

General Execution Rules:
- When using tools, you must collect all necessary data before formulating your final response. You can call tools sequentially across loop iterations.
- Format all final answers beautifully using clean Markdown (headers, bold text, bullet points). Do not wrap the final output in a markdown code block.
- Be conversational and context-aware. Remember the details of the trip previously discussed in the conversation history.
"""
user_message = """"
I'm planning a trip to Zurich. Find me a hotel under $150 and let me know if I need to pack an umbrella or a heavy coat based on the local forecast.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [23]:
response = travel_agent(messages)

In [24]:
display(Markdown(response))

Here's everything you need to know for your Zurich trip! 🌧️🧳

---

## 🏨 Hotels in Zurich under $150/night

| Hotel | Price/Night |
|-------|------------|
| **Zurich Capsule Stay** | 💲 **$65** – A budget-friendly option! |
| **Swiss Business Inn** | 💲 **$130** – Great value in your price range. |

---

## 🌤️ Zurich Weather Update

| Temperature | Condition |
|------------|-----------|
| **14°C (57°F)** | 🌧️ **Heavy Rain** |

---

## 🎒 What to Pack?

| Item | Verdict |
|------|---------|
| ☂️ **Umbrella** | **Absolutely YES!** Heavy rain is in the forecast — you'll want it. |
| 🧥 **Heavy Coat** | **Not necessary.** At 14°C (57°F), a **light waterproof jacket** or **raincoat** will do the trick. A heavy winter coat would be overkill. |

---

**Bottom line:** Grab that umbrella and a light rain jacket, and you'll be set for a cozy Zurich adventure! Would you like more details on either hotel or any other trip planning help? 😊

In [25]:
messages.append({"role": "system", "content": response})

# Append your new request as a user message
messages.append({
    "role": "user", 
    "content": "In which country this city is located in? What currency do people use there?"
})

second_response = travel_agent(messages)
display(Markdown(second_response))

Great questions! Here's the info on Zurich's location and currency:

---

## 🇨🇭 Where is Zurich?

Zurich is located in **Switzerland** — a beautiful country in Central Europe known for its Alps, chocolate, and precision watches.

---

## 💰 What currency do they use?

Switzerland uses the **Swiss Franc (CHF)**, **not** the Euro! Here's a quick overview:

| Currency | Symbol | Code | 
|----------|--------|------|
| **Swiss Franc** | **Fr.** or **SFr.** | **CHF** |

**Quick tip:** While some touristy spots near the border *might* accept Euros, you'll almost always get poor exchange rates. Best to use **Swiss Francs** or a **credit card** — cards are widely accepted across Switzerland! 🏔️

---

So to recap your Zurich trip details so far:
- **📍 Country:** Switzerland
- **💰 Currency:** Swiss Franc (CHF)
- **🏨 Hotels under $150:** Zurich Capsule Stay ($65/night) or Swiss Business Inn ($130/night)
- **🌧️ Weather:** 14°C with heavy rain — bring that **umbrella** and a **light waterproof jacket**!

Need anything else for your trip planning? 😊

In [26]:
judge = f"""You are judging an Autonomous Travel Concierge Agent's response to a user's query regarding their trip planning?

User has asked which country does Zurich belongs to and what currency is being used there?

Your job is to evaluate the agent's response for clarity and accuracy.

Here is the response from the agent:

{second_response}

Now respond with a short message and reasoning whether the agent was clear and accuracte with it's response based on the user's query, and if the response is relevant to the user's query. Do not include markdown formatting or code blocks."""

In [29]:
show(judge)

You are judging an Autonomous Travel Concierge Agent's response to a user's query regarding their trip planning?

User has asked which country does Zurich belongs to and what currency is being used there?

Your job is to evaluate the agent's response for clarity and accuracy.

Here is the response from the agent:

Great questions! Here's the info on Zurich's location and currency:

---

## 🇨🇭 Where is Zurich?

Zurich is located in **Switzerland** — a beautiful country in Central Europe known for its Alps, chocolate, and 
precision watches.

---

## 💰 What currency do they use?

Switzerland uses the **Swiss Franc (CHF)**, **not** the Euro! Here's a quick overview:

| Currency | Symbol | Code | 
|----------|--------|------|
| **Swiss Franc** | **Fr.** or **SFr.** | **CHF** |

**Quick tip:** While some touristy spots near the border *might* accept Euros, you'll almost always get poor 
exchange rates. Best to use **Swiss Francs** or a **credit card** — cards are widely accepted across Switzerland! 🏔️

---

So to recap your Zurich trip details so far:
- **📍 Country:** Switzerland
- **💰 Currency:** Swiss Franc (CHF)
- **🏨 Hotels under $150:** Zurich Capsule Stay ($65/night) or Swiss Business Inn ($130/night)
- **🌧️ Weather:** 14°C with heavy rain — bring that **umbrella** and a **light waterproof jacket**!

Need anything else for your trip planning? 😊

Now respond with a short message and reasoning whether the agent was clear and accuracte with it's response based 
on the user's query, and if the response is relevant to the user's query. Do not include markdown formatting or 
code blocks.

In [30]:
judge_messages = [{"role": "user", "content": judge}]

In [32]:

gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
response = gemini.chat.completions.create(
    model="gemini-2.5-flash",
    messages=judge_messages,
)
results = response.choices[0].message.content
show(results)

The agent was highly clear and accurate in its response regarding Zurich's country and currency. Both pieces of 
information were presented correctly and clearly, with valuable additional context and tips, especially regarding 
the Swiss Franc versus the Euro.

Regarding relevance, the agent's answers to the direct questions about country and currency were perfectly 
relevant. However, the inclusion of information about hotels and weather was not requested by the user, making 
those parts of the response less relevant to the original query. While potentially helpful, they went beyond what 
was asked.